# ISTAT: fonti italiane per dettaglio granulare

            Questo foglio organizza le fonti italiane candidate rispetto al bisogno del progetto: aprire il dettaglio sulle imprese italiane oltre le classi internazionali standard.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from valore_aggiunto_imprese.analysis import (
    METRIC_LABELS,
    add_source_note,
    latest_common_year_with_values,
    latest_year_with_values,
    SIZE_ORDER_BUSINESS,
    SIZE_ORDER_OFFICIAL,
    add_share,
    enrich_business_demography,
    enrich_sbs,
    read_project_csv,
)

pd.options.display.max_columns = 80
pd.options.display.float_format = "{:,.2f}".format

## Inventario fonti

In [ ]:
istat = read_project_csv("data/processed/istat_sources_inventory.csv")
istat

## Ruolo analitico delle fonti

In [ ]:
roles = {
    "Frame SBS": {
        "area": "struttura imprese",
        "uso": "dettaglio su unita economiche e classi dimensionali",
    },
    "Frame Territoriale": {"area": "territorio", "uso": "lettura regionale e locale"},
    "Frame Territoriale Anticipato": {
        "area": "territorio",
        "uso": "aggiornamento anticipato del dettaglio locale",
    },
    "Conti economici delle imprese": {
        "area": "conto economico",
        "uso": "valore aggiunto e performance economica",
    },
    "Microdati struttura e performance imprese": {
        "area": "microdati",
        "uso": "analisi fine se accessibile e documentata",
    },
}
source_map = istat.assign(
    area=istat["candidate_source"].map(lambda x: roles.get(x, {}).get("area", "da classificare")),
    uso=istat["candidate_source"].map(lambda x: roles.get(x, {}).get("uso", "da classificare")),
)
source_map[["candidate_source", "area", "uso", "method_status"]]

## Fonti candidate per area

In [ ]:
area_summary = source_map.groupby("area", as_index=False).size().sort_values(
    "size", ascending=True
)

fig = px.bar(
    area_summary,
    x="size",
    y="area",
    orientation="h",
    labels={"size": "Numero fonti", "area": "Area"},
    title="Fonti ISTAT candidate per area analitica",
)
add_source_note(fig, "ISTAT; configurazione del progetto")
fig

## Collegamento con i dataset internazionali

In [ ]:
pd.DataFrame(
    [
        {
            "bisogno": "Aprire la classe 0-9",
            "fonte_istat": "Frame SBS / microdati",
            "dataset_internazionale": "Eurostat SBS",
        },
        {
            "bisogno": "Descrivere micro-classi",
            "fonte_istat": "Frame SBS",
            "dataset_internazionale": "Business Demography",
        },
        {
            "bisogno": "Aggiungere territorio",
            "fonte_istat": "Frame Territoriale",
            "dataset_internazionale": "Eurostat SBS",
        },
        {
            "bisogno": "Collegare valore aggiunto",
            "fonte_istat": "Conti economici delle imprese",
            "dataset_internazionale": "Eurostat SBS",
        },
    ]
)